# Fine-Tuning Whisper Small for Danish ASR on CORAL

This notebook documents the development of a Danish automatic speech recognition pipeline based on `openai/whisper-small` and the CORAL speech dataset.
The main goal is to see whether fine-tuning improves Danish transcription quality compared with the base Whisper Small model.


In [1]:
import io
import math
from pathlib import Path

import pandas as pd
import librosa
import soundfile as sf
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import jiwer


## Problem And Goal

The task is Danish speech-to-text transcription.

I used the CORAL dataset to fine-tune Whisper Small and evaluated the model with word error rate (WER), with some sections also reporting character error rate (CER).

The aim is not just to train a model once, but to show:
- that the training pipeline works end to end
- that the fine-tuned model can be compared against the base model
- that the final results are stable enough to be worth using


In [2]:
DATA_ROOT = Path("/workspace/coral_30gb")
METADATA_PATH = DATA_ROOT / "metadata.csv"
MODEL_NAME = "openai/whisper-small"
LANGUAGE = "Danish"
TASK = "transcribe"
TARGET_SR = 16000

NUM_EPOCHS = 3
LEARNING_RATE = 1e-5
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
VAL_FRACTION = 0.02
SEED = 42
MAX_EVAL_SAMPLES = 512


In [3]:
df = pd.read_csv(METADATA_PATH)
print(df["split"].value_counts())
print(len(df))
df

split
train    59711
Name: count, dtype: int64
59711


,split,id,text,audio_path,bytes
0,train,rec_30141c4db5e1ab5e60edbd662f226755,han drog til paris for yderligere at dygtiggør...,/workspace/coral_30gb/audio/train/rec_30141c4d...,437838
1,train,rec_6677f5f759f1e09a12e43bda530f77d9,senere arbejdede han i dresden og hannover,/workspace/coral_30gb/audio/train/rec_6677f5f7...,270798
2,train,rec_7686fe5ffecf9247301c3ab7f6df8730,filmen er baseret på en roman af fletcher kneb...,/workspace/coral_30gb/audio/train/rec_7686fe5f...,529998
3,train,rec_daf33e4bd6b8a7f753f831775fc2c2a5,menneskesønnen repræsenterer altså også deres ...,/workspace/coral_30gb/audio/train/rec_daf33e4b...,616398
4,train,rec_327824a75063cef4f63ac77071618b76,hvis en evakuering af stationen havde været nø...,/workspace/coral_30gb/audio/train/rec_327824a7...,483918
...,...,...,...,...,...
59706,train,rec_48e48a5076633d3c2de9b4831fd7fb4e,senere det år rejste han tilbage til tjekkoslo...,/workspace/coral_30gb/audio/train/rec_48e48a50...,553038
59707,train,rec_ff16682ef34b20777f641f02303359a4,lovecraft fanzine redigeret af thomas winther,/workspace/coral_30gb/audio/train/rec_ff16682e...,437838
59708,train,rec_cfd0614c3d229decf67f55d668eebd6b,byen ligger på jernbanelinien fra verona via b...,/workspace/coral_30gb/audio/train/rec_cfd0614c...,633678
59709,train,rec_ff3d7063dac646c1271d7c96c40119e4,med bayern nåede han at vinde fire tyske meste...,/workspace/coral_30gb/audio/train/rec_ff3d7063...,581838


### Model Setup

Here the base Whisper Small model and processor are initialized for Danish transcription.

This section also sets the generation configuration used during fine-tuning, including language and task settings.


In [4]:
train_df = df[df["split"] == "train"].copy()

val_df = train_df.sample(frac=VAL_FRACTION, random_state=SEED)
train_df = train_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

if len(val_df) > MAX_EVAL_SAMPLES:
    val_df = val_df.sample(n=MAX_EVAL_SAMPLES, random_state=SEED).reset_index(drop=True)

print("train rows:", len(train_df))
print("val rows:", len(val_df))


train rows: 58517
val rows: 512


In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language=LANGUAGE, task=TASK)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    low_cpu_mem_usage=True,
).to(device)

forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
model.generation_config.forced_decoder_ids = forced_decoder_ids
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.config.use_cache = False
model.gradient_checkpointing_enable()

decoder_start_token_id = processor.tokenizer.bos_token_id

print(device, dtype, MODEL_NAME)


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

cuda torch.bfloat16 openai/whisper-small


In [6]:
def load_audio_array(audio_path):
    audio_array, sampling_rate = sf.read(audio_path)

    if getattr(audio_array, "ndim", 1) > 1:
        audio_array = audio_array.mean(axis=1)

    if sampling_rate != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=TARGET_SR)
        sampling_rate = TARGET_SR

    return audio_array, sampling_rate


In [7]:
class LocalWhisperDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "id": row["id"],
            "audio": row["audio_path"],
            "sentence": row["text"],
        }

train_dataset = LocalWhisperDataset(train_df)
val_dataset = LocalWhisperDataset(val_df)

print(len(train_dataset), len(val_dataset))


58517 512


In [8]:
def collate_batch(features):
    audio_arrays = []
    label_features = []

    for feature in features:
        text = str(feature["sentence"]).strip()
        if not text:
            continue

        audio_array, _ = load_audio_array(feature["audio"])
        audio_arrays.append(audio_array)
        label_features.append({"input_ids": processor.tokenizer(text).input_ids})

    batch = processor.feature_extractor(
        audio_arrays,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )
    batch["input_features"] = batch["input_features"].to(dtype=dtype)

    labels_batch = processor.tokenizer.pad(label_features, return_tensors="pt")
    labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

    if decoder_start_token_id is not None and (labels[:, 0] == decoder_start_token_id).all().cpu().item():
        labels = labels[:, 1:]

    batch["labels"] = labels
    return batch


In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

print("steps per epoch:", math.ceil(len(train_loader) / GRAD_ACCUM_STEPS))


steps per epoch: 1829


In [10]:
@torch.no_grad()
def evaluate_model():
    model.eval()

    predictions = []
    references = []

    for batch in val_loader:
        input_features = batch["input_features"].to(device)
        label_ids = batch["labels"].clone()

        generated_ids = model.generate(input_features=input_features, max_length=225)

        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        pred_text = processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
        ref_text = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

        predictions.extend(pred_text)
        references.extend(ref_text)

    wer = jiwer.wer(references, predictions)
    return {
        "wer": wer,
        "predictions": predictions[:5],
        "references": references[:5],
    }


In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

use_fp16_scaler = (dtype == torch.float16 and device.type == "cuda")
scaler = torch.cuda.amp.GradScaler() if use_fp16_scaler else None


### Training Loop

This cell runs the fine-tuning loop.

The objective here is to confirm that optimization works, loss decreases, and the model can be evaluated on a held-out validation subset.


In [12]:
autocast_dtype = torch.bfloat16 if dtype == torch.bfloat16 else torch.float16

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad()
    running_loss = 0.0
    optimizer_steps = 0

    for step, batch in enumerate(train_loader, start=1):
        input_features = batch["input_features"].to(device)
        labels = batch["labels"].to(device)

        with torch.autocast(device_type="cuda", dtype=autocast_dtype, enabled=(device.type == "cuda")):
            outputs = model(input_features=input_features, labels=labels)
            loss = outputs.loss / GRAD_ACCUM_STEPS

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        running_loss += loss.item()

        if step % GRAD_ACCUM_STEPS == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad()
            optimizer_steps += 1

            if optimizer_steps % 20 == 0:
                print({
                    "epoch": epoch + 1,
                    "step": optimizer_steps,
                    "loss": running_loss / 20,
                })
                running_loss = 0.0

    metrics = evaluate_model()
    print({
        "epoch": epoch + 1,
        "val_wer": metrics["wer"],
    })


/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


{'epoch': 1, 'step': 20, 'loss': 2.389375573396683}
{'epoch': 1, 'step': 40, 'loss': 1.577328473329544}
{'epoch': 1, 'step': 60, 'loss': 1.2433191329240798}
{'epoch': 1, 'step': 80, 'loss': 1.1440533325076103}
{'epoch': 1, 'step': 100, 'loss': 1.0480201572179795}
{'epoch': 1, 'step': 120, 'loss': 1.0047689899802208}
{'epoch': 1, 'step': 140, 'loss': 1.0011854127049447}
{'epoch': 1, 'step': 160, 'loss': 0.9862035498023033}
{'epoch': 1, 'step': 180, 'loss': 0.9159195810556412}
{'epoch': 1, 'step': 200, 'loss': 0.8524365842342376}
{'epoch': 1, 'step': 220, 'loss': 0.847070449590683}
{'epoch': 1, 'step': 240, 'loss': 0.8380159825086594}
{'epoch': 1, 'step': 260, 'loss': 0.8325177818536759}
{'epoch': 1, 'step': 280, 'loss': 0.8092324957251549}
{'epoch': 1, 'step': 300, 'loss': 0.7428988352417946}
{'epoch': 1, 'step': 320, 'loss': 0.7322253331542015}
{'epoch': 1, 'step': 340, 'loss': 0.7243525564670563}
{'epoch': 1, 'step': 360, 'loss': 0.6774992063641548}
{'epoch': 1, 'step': 380, 'loss': 0

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

{'epoch': 1, 'val_wer': 0.2713958002470443}
{'epoch': 2, 'step': 20, 'loss': 0.4487687170505524}
{'epoch': 2, 'step': 40, 'loss': 0.45790529772639277}
{'epoch': 2, 'step': 60, 'loss': 0.49806898459792137}
{'epoch': 2, 'step': 80, 'loss': 0.49062447175383567}
{'epoch': 2, 'step': 100, 'loss': 0.4721032314002514}
{'epoch': 2, 'step': 120, 'loss': 0.4675634130835533}
{'epoch': 2, 'step': 140, 'loss': 0.46025056913495066}
{'epoch': 2, 'step': 160, 'loss': 0.49724818468093873}
{'epoch': 2, 'step': 180, 'loss': 0.48047718331217765}
{'epoch': 2, 'step': 200, 'loss': 0.5049330294132233}
{'epoch': 2, 'step': 220, 'loss': 0.45731955394148827}
{'epoch': 2, 'step': 240, 'loss': 0.48040038496255877}
{'epoch': 2, 'step': 260, 'loss': 0.47393063083291054}
{'epoch': 2, 'step': 280, 'loss': 0.46127164363861084}
{'epoch': 2, 'step': 300, 'loss': 0.46605021730065344}
{'epoch': 2, 'step': 320, 'loss': 0.48162889406085013}
{'epoch': 2, 'step': 340, 'loss': 0.46838048845529556}
{'epoch': 2, 'step': 360, 'lo

In [13]:
metrics = evaluate_model()
print({"final_val_wer": metrics["wer"]})
print("Reference:", metrics["references"][0])
print("Prediction:", metrics["predictions"][0])


{'final_val_wer': 0.25304393859184754}
Reference: diabolus in musica er det syvende studiealbum af det amerikanske thrash metalband slayer
Prediction: de aplussen musiker er det syvende studiealbum af det amerikanske træs metalband slayer


In [14]:
output_dir = Path("/workspace/outputs/whisper-small-coral-30gb")
output_dir.mkdir(parents=True, exist_ok=True)

model.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

print(f"Saved to {output_dir}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to /workspace/outputs/whisper-small-coral-30gb


In [27]:
import pandas as pd
from pathlib import Path

TEST_METADATA_PATH = Path("/workspace/coral_30gb/metadata_test.csv")
test_df = pd.read_csv(TEST_METADATA_PATH)

print(len(test_df))
print(test_df["split"].value_counts())
test_df.head()



1997
split
test    1997
Name: count, dtype: int64


,split,id,text,audio_path,bytes
0,test,rec_45064562a057476e51179297fb172057,normannerne er en dansk film fra 1976,/workspace/coral_30gb/audio/test/rec_45064562a...,483918
1,test,rec_becc6eb53b441b31116584cd5c4e1c51,begge klynger støttes af klyngeorganisationer ...,/workspace/coral_30gb/audio/test/rec_becc6eb53...,702798
2,test,rec_e59f62c2675fea5f9e7e726b51283250,i mildere tilfælde kan smerterne klares med pa...,/workspace/coral_30gb/audio/test/rec_e59f62c26...,777678
3,test,rec_a47501841f128c1e6eb7fdd21647aa14,grant at ødelægge lees hær gennem udmattelse v...,/workspace/coral_30gb/audio/test/rec_a47501841...,956238
4,test,rec_164d762f9438572ebe7b3bb73b2583b6,i 1999 blev herning rejser fusioneret med falk...,/workspace/coral_30gb/audio/test/rec_164d762f9...,645198


In [28]:
from torch.utils.data import Dataset, DataLoader

class LocalWhisperDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "id": row["id"],
            "audio": row["audio_path"],
            "sentence": row["text"],
        }

test_dataset = LocalWhisperDataset(test_df)
print(len(test_dataset))


1997


### Part 1 Result

This section reports the validation result from the prototype run and shows a sample reference/prediction pair.

The goal is to check whether the pipeline produces plausible Danish transcriptions before moving to a more rigorous experiment setup.


In [29]:
test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_batch,
    num_workers=2,
    pin_memory=True,
)

print("test batches:", len(test_loader))


test batches: 500


In [30]:
base_model, base_processor = load_whisper_model_and_processor(BASE_MODEL_NAME)

base_test_results = evaluate_model_pair(
    model=base_model,
    processor=base_processor,
    val_loader=test_loader,
    device=device,
)

print({"base_test_wer": base_test_results["wer"]})
print("Reference:", base_test_results["references"][0])
print("Base prediction:", base_test_results["predictions"][0])


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

{'base_test_wer': 0.6313470901924415}
Reference: normannerne er en dansk film fra 1976
Base prediction:  og nu mener jeg, at denne film for 1906 er fjerns.


In [31]:
finetuned_model, finetuned_processor = load_whisper_model_and_processor(FINETUNED_DIR)

finetuned_test_results = evaluate_model_pair(
    model=finetuned_model,
    processor=finetuned_processor,
    val_loader=test_loader,
    device=device,
)

print({"finetuned_test_wer": finetuned_test_results["wer"]})
print("Reference:", finetuned_test_results["references"][0])
print("Finetuned prediction:", finetuned_test_results["predictions"][0])


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

{'finetuned_test_wer': 0.3973104567586367}
Reference: normannerne er en dansk film fra 1976
Finetuned prediction: anomandre er en dansk film fra 1976


In [32]:
base_test_wer = base_test_results["wer"]
finetuned_test_wer = finetuned_test_results["wer"]

relative_test_improvement = (
    (base_test_wer - finetuned_test_wer) / base_test_wer
    if base_test_wer > 0 else 0.0
)

print({
    "base_test_wer": base_test_wer,
    "finetuned_test_wer": finetuned_test_wer,
    "absolute_gain": base_test_wer - finetuned_test_wer,
    "relative_improvement": relative_test_improvement,
})



{'base_test_wer': 0.6313470901924415, 'finetuned_test_wer': 0.3973104567586367, 'absolute_gain': 0.23403663343380482, 'relative_improvement': 0.370694087403599}


In [33]:
rows = []
for ref, base_pred, ft_pred in zip(
    base_test_results["references"],
    base_test_results["predictions"],
    finetuned_test_results["predictions"],
):
    rows.append({
        "reference": ref,
        "base_prediction": base_pred,
        "finetuned_prediction": ft_pred,
    })

test_comparison_df = pd.DataFrame(rows)
test_comparison_path = Path("/workspace/outputs/whisper-small-coral-30gb/test_comparison.csv")
test_comparison_df.to_csv(test_comparison_path, index=False)

print(test_comparison_path)
test_comparison_df.head(10)



/workspace/outputs/whisper-small-coral-30gb/test_comparison.csv


,reference,base_prediction,finetuned_prediction
0,normannerne er en dansk film fra 1976,"og nu mener jeg, at denne film for 1906 er fj...",anomandre er en dansk film fra 1976
1,begge klynger støttes af klyngeorganisationer ...,Bav og klunger støtter sig klungorganisatione...,bevr klunger støttes af klungorganisationer ti...
2,i mildere tilfælde kan smerterne klares med pa...,I midlere tilfælde kan jeg smadre klare med P...,i midlertifaldet kan er smatten og klars med p...
3,grant at ødelægge lees hær gennem udmattelse v...,Grønt er du der laver lige her og gennem og u...,grønse er et ødelag liges her og gennem og uma...
4,i 1999 blev herning rejser fusioneret med falk...,I 1999 blev Herno Reise funktionert med Folk ...,i 1999 blev herning rejse fusioneret med folk ...
5,det blev til yderligere 10 mål i den første sæ...,"Det blev til ørelætik mål i den første sæson,...",det blev til øre letikmål i den første sæson o...
6,fyrtårnet har været byggnadsminne siden 1933,Fyre tagerne har været bygnesmænden sin nænde...,førtårn har været bygnersmænd siden 1933
7,i danmark har industrialiseringen været afgøre...,I Danmark har indåsdragisering været afgærend...,i danmark har industrialiseringen været afgøre...
8,horakova blev født milada kralova i prag,"Og hvorfor blev det for, at der blev en lille...",horåbov er blevet frejt med lærder kalovar i p...
9,på alle danske sirenestationer anvendtes samme...,og alle den syreenerstation anvendes sammen p...,og alle danske sirene stationer anvandles samm...


## Part 2 - Final Multi-Seed Experiment

This is the main experimental section of the notebook.

Instead of relying on one training run, this workflow repeats training across multiple seeds and compares the fine-tuned checkpoints against the base Whisper Small model on the test set.

This makes the final conclusion more reliable and less dependent on one lucky initialization or split.


In [37]:
df = pd.read_csv(DATA_ROOT / "metadata.csv")
test_df = pd.read_csv(TEST_METADATA_PATH)

full_train_df = df[df["split"] == "train"].copy()

val_df = full_train_df.sample(frac=VAL_FRACTION, random_state=VAL_SPLIT_SEED)
train_df = full_train_df.drop(val_df.index).reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

if len(val_df) > MAX_EVAL_SAMPLES:
    val_df = val_df.sample(n=MAX_EVAL_SAMPLES, random_state=VAL_SPLIT_SEED).reset_index(drop=True)

print("train rows:", len(train_df))
print("val rows:", len(val_df))
print("test rows:", len(test_df))


train rows: 58517
val rows: 512
test rows: 1997


### Experimental Setup

This section rebuilds the dataset splits and prepares the inputs for the final experiment.

Compared with the first run, the focus here is on reproducibility and consistency across repeated runs.


In [38]:
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_audio_array(audio_path):
    audio_array, sampling_rate = sf.read(audio_path)

    if getattr(audio_array, "ndim", 1) > 1:
        audio_array = audio_array.mean(axis=1)

    if sampling_rate != TARGET_SR:
        audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=TARGET_SR)
        sampling_rate = TARGET_SR

    return audio_array, sampling_rate


class LocalWhisperDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "id": str(row["id"]),
            "audio": row["audio_path"],
            "sentence": str(row["text"]).strip(),
        }


def load_model_and_processor(model_path):
    model_dtype = (
        torch.bfloat16
        if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else (torch.float16 if torch.cuda.is_available() else torch.float32)
    )

    processor = WhisperProcessor.from_pretrained(model_path, language=LANGUAGE, task=TASK)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_path,
        torch_dtype=model_dtype,
        low_cpu_mem_usage=True,
    ).to(device)

    forced_decoder_ids = processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)
    model.generation_config.forced_decoder_ids = forced_decoder_ids
    model.generation_config.language = LANGUAGE
    model.generation_config.task = TASK
    model.config.use_cache = False
    model.gradient_checkpointing_enable()

    return model, processor, model_dtype


def make_collate_fn(processor, feature_dtype):
    decoder_start_token_id = processor.tokenizer.bos_token_id

    def collate_batch(features):
        ids = []
        audio_arrays = []
        label_features = []

        for feature in features:
            text = str(feature["sentence"]).strip()
            if not text:
                continue

            audio_array, _ = load_audio_array(feature["audio"])
            ids.append(feature["id"])
            audio_arrays.append(audio_array)
            label_features.append({"input_ids": processor.tokenizer(text).input_ids})

        if not audio_arrays:
            raise ValueError("Encountered an empty batch after filtering blank transcripts.")

        batch = processor.feature_extractor(
            audio_arrays,
            sampling_rate=TARGET_SR,
            return_tensors="pt",
        )
        batch["input_features"] = batch["input_features"].to(dtype=feature_dtype)

        labels_batch = processor.tokenizer.pad(label_features, return_tensors="pt")
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if decoder_start_token_id is not None and (labels[:, 0] == decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        batch["ids"] = ids
        return batch

    return collate_batch


def build_loader(dataset_df, processor, feature_dtype, batch_size, shuffle, seed=None):
    generator = None
    if shuffle:
        generator = torch.Generator()
        generator.manual_seed(seed if seed is not None else 0)

    return DataLoader(
        LocalWhisperDataset(dataset_df),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=make_collate_fn(processor, feature_dtype),
        num_workers=2,
        pin_memory=pin_memory,
        generator=generator,
    )


@torch.inference_mode()
def evaluate_model_pair(model, processor, data_loader):
    model.eval()
    predictions = []
    references = []
    rows = []

    for batch in data_loader:
        input_features = batch["input_features"].to(device)
        label_ids = batch["labels"].clone()

        generated_ids = model.generate(input_features=input_features, max_length=225)
        label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

        pred_texts = [text.strip() for text in processor.tokenizer.batch_decode(generated_ids, skip_special_tokens=True)]
        ref_texts = [text.strip() for text in processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)]

        predictions.extend(pred_texts)
        references.extend(ref_texts)

        for sample_id, ref_text, pred_text in zip(batch["ids"], ref_texts, pred_texts):
            rows.append(
                {
                    "id": sample_id,
                    "reference": ref_text,
                    "prediction": pred_text,
                }
            )

    return {
        "wer": jiwer.wer(references, predictions),
        "cer": jiwer.cer(references, predictions),
        "rows": rows,
    }


def paired_bootstrap_ci(base_rows, finetuned_rows, metric_fn, n_boot=2000, confidence_level=0.95, seed=42):
    assert len(base_rows) == len(finetuned_rows), "Mismatched prediction lengths"

    for base_row, ft_row in zip(base_rows, finetuned_rows):
        assert base_row["id"] == ft_row["id"], "Mismatched sample ids"
        assert base_row["reference"] == ft_row["reference"], "Mismatched references"

    references = [row["reference"] for row in base_rows]
    base_predictions = [row["prediction"] for row in base_rows]
    ft_predictions = [row["prediction"] for row in finetuned_rows]

    base_score = metric_fn(references, base_predictions)
    ft_score = metric_fn(references, ft_predictions)

    rng = np.random.default_rng(seed)
    deltas = np.empty(n_boot, dtype=np.float64)

    for i in range(n_boot):
        idx = rng.integers(0, len(references), size=len(references))
        sample_refs = [references[j] for j in idx]
        sample_base = [base_predictions[j] for j in idx]
        sample_ft = [ft_predictions[j] for j in idx]
        deltas[i] = metric_fn(sample_refs, sample_base) - metric_fn(sample_refs, sample_ft)

    alpha = 1.0 - confidence_level
    ci_lower = float(np.quantile(deltas, alpha / 2.0))
    ci_upper = float(np.quantile(deltas, 1.0 - alpha / 2.0))

    return {
        "base_score": float(base_score),
        "finetuned_score": float(ft_score),
        "absolute_gain": float(base_score - ft_score),
        "relative_improvement": float((base_score - ft_score) / base_score) if base_score > 0 else 0.0,
        "ci_lower": ci_lower,
        "ci_upper": ci_upper,
        "probability_gain_gt_zero": float(np.mean(deltas > 0.0)),
    }


In [39]:
def train_one_seed(seed: int):
    set_seed(seed)

    seed_dir = RUN_ROOT / f"seed-{seed}"
    checkpoint_dir = seed_dir / "best_checkpoint"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model, processor, model_dtype = load_model_and_processor(MODEL_NAME)

    train_loader = build_loader(
        train_df,
        processor=processor,
        feature_dtype=model_dtype,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=seed,
    )
    val_loader = build_loader(
        val_df,
        processor=processor,
        feature_dtype=model_dtype,
        batch_size=4,
        shuffle=False,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    use_fp16_scaler = model_dtype == torch.float16 and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler() if use_fp16_scaler else None
    autocast_dtype = torch.bfloat16 if model_dtype == torch.bfloat16 else torch.float16

    best_val_wer = float("inf")
    history = []

    for epoch in range(NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()
        running_loss = 0.0
        running_updates = 0
        accum_steps = 0

        for step, batch in enumerate(train_loader, start=1):
            input_features = batch["input_features"].to(device)
            labels = batch["labels"].to(device)

            with torch.autocast(
                device_type="cuda",
                dtype=autocast_dtype,
                enabled=(device.type == "cuda"),
            ):
                outputs = model(input_features=input_features, labels=labels)
                loss = outputs.loss / GRAD_ACCUM_STEPS

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            running_loss += loss.item()
            accum_steps += 1
            should_step = (accum_steps == GRAD_ACCUM_STEPS) or (step == len(train_loader))

            if should_step:
                if scaler is not None:
                    scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                if scaler is not None:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()

                optimizer.zero_grad()
                running_updates += 1
                accum_steps = 0

        val_metrics = evaluate_model_pair(model, processor, val_loader)
        epoch_record = {
            "epoch": epoch + 1,
            "train_loss": running_loss / max(running_updates, 1),
            "val_wer": val_metrics["wer"],
            "val_cer": val_metrics["cer"],
        }
        history.append(epoch_record)
        print({"seed": seed, **epoch_record})

        if val_metrics["wer"] < best_val_wer:
            best_val_wer = val_metrics["wer"]
            model.save_pretrained(checkpoint_dir)
            processor.save_pretrained(checkpoint_dir)

    summary = {
        "seed": seed,
        "best_val_wer": min(x["val_wer"] for x in history),
        "best_val_cer": min(x["val_cer"] for x in history),
        "best_checkpoint_dir": str(checkpoint_dir),
        "history": history,
    }

    (seed_dir / "training_summary.json").write_text(
        json.dumps(summary, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    del model, processor, train_loader, val_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary


In [40]:
seed_summaries = []

for seed in TRAINING_SEEDS:
    seed_summaries.append(train_one_seed(seed))

multi_seed_train_df = pd.DataFrame(
    [
        {
            "seed": s["seed"],
            "best_val_wer": s["best_val_wer"],
            "best_val_cer": s["best_val_cer"],
            "best_checkpoint_dir": s["best_checkpoint_dir"],
        }
        for s in seed_summaries
    ]
).sort_values("seed")

display(multi_seed_train_df)

print("val WER mean:", multi_seed_train_df["best_val_wer"].mean())
print("val WER std :", multi_seed_train_df["best_val_wer"].std(ddof=1))
print("val CER mean:", multi_seed_train_df["best_val_cer"].mean())
print("val CER std :", multi_seed_train_df["best_val_cer"].std(ddof=1))


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


{'seed': 13, 'epoch': 1, 'train_loss': 0.6410001476258121, 'val_wer': 0.2724545614963826, 'val_cer': 0.08944749633294026}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'seed': 13, 'epoch': 2, 'train_loss': 0.4649445268342311, 'val_wer': 0.25816128463031585, 'val_cer': 0.08504702464839368}


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'seed': 123, 'epoch': 3, 'train_loss': 0.4339783968268489, 'val_wer': 0.2555143815069702, 'val_cer': 0.08708907360004602}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,seed,best_val_wer,best_val_cer,best_checkpoint_dir
0,13,0.250926,0.082919,/workspace/outputs/whisper-small-coral-30gb-mu...
1,42,0.251279,0.083810,/workspace/outputs/whisper-small-coral-30gb-mu...
2,123,0.255514,0.086801,/workspace/outputs/whisper-small-coral-30gb-mu...


val WER mean: 0.25257337803658614
val WER std : 0.002553089161132961
val CER mean: 0.08451014792871044
val CER std : 0.002033795481664024


In [41]:
base_model, base_processor, base_dtype = load_model_and_processor(MODEL_NAME)
base_test_loader = build_loader(
    test_df,
    processor=base_processor,
    feature_dtype=base_dtype,
    batch_size=4,
    shuffle=False,
)
base_test_results = evaluate_model_pair(base_model, base_processor, base_test_loader)

del base_model, base_processor, base_test_loader
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

test_rows = []

for summary in seed_summaries:
    seed = summary["seed"]
    checkpoint_dir = Path(summary["best_checkpoint_dir"])

    ft_model, ft_processor, ft_dtype = load_model_and_processor(checkpoint_dir)
    ft_test_loader = build_loader(
        test_df,
        processor=ft_processor,
        feature_dtype=ft_dtype,
        batch_size=4,
        shuffle=False,
    )
    ft_test_results = evaluate_model_pair(ft_model, ft_processor, ft_test_loader)

    wer_bootstrap = paired_bootstrap_ci(
        base_rows=base_test_results["rows"],
        finetuned_rows=ft_test_results["rows"],
        metric_fn=jiwer.wer,
        n_boot=BOOTSTRAP_SAMPLES,
        confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
        seed=BOOTSTRAP_SEED,
    )
    cer_bootstrap = paired_bootstrap_ci(
        base_rows=base_test_results["rows"],
        finetuned_rows=ft_test_results["rows"],
        metric_fn=jiwer.cer,
        n_boot=BOOTSTRAP_SAMPLES,
        confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
        seed=BOOTSTRAP_SEED,
    )

    test_rows.append(
        {
            "seed": seed,
            "base_test_wer": wer_bootstrap["base_score"],
            "finetuned_test_wer": wer_bootstrap["finetuned_score"],
            "wer_absolute_gain": wer_bootstrap["absolute_gain"],
            "wer_relative_improvement": wer_bootstrap["relative_improvement"],
            "wer_ci_lower": wer_bootstrap["ci_lower"],
            "wer_ci_upper": wer_bootstrap["ci_upper"],
            "wer_prob_gain_gt_zero": wer_bootstrap["probability_gain_gt_zero"],
            "base_test_cer": cer_bootstrap["base_score"],
            "finetuned_test_cer": cer_bootstrap["finetuned_score"],
            "cer_absolute_gain": cer_bootstrap["absolute_gain"],
            "cer_ci_lower": cer_bootstrap["ci_lower"],
            "cer_ci_upper": cer_bootstrap["ci_upper"],
        }
    )

    del ft_model, ft_processor, ft_test_loader
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

multi_seed_test_df = pd.DataFrame(test_rows).sort_values("seed")
display(multi_seed_test_df)

print("test WER mean:", multi_seed_test_df["finetuned_test_wer"].mean())
print("test WER std :", multi_seed_test_df["finetuned_test_wer"].std(ddof=1))
print("mean WER gain:", multi_seed_test_df["wer_absolute_gain"].mean())
print("seeds with WER CI > 0:", int((multi_seed_test_df["wer_ci_lower"] > 0).sum()))


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

,seed,base_test_wer,finetuned_test_wer,wer_absolute_gain,wer_relative_improvement,wer_ci_lower,wer_ci_upper,wer_prob_gain_gt_zero,base_test_cer,finetuned_test_cer,cer_absolute_gain,cer_ci_lower,cer_ci_upper
0,13,0.631347,0.393740,0.237607,0.376350,0.225130,0.253183,1.0,0.253809,0.157616,0.096194,0.089136,0.106222
1,42,0.631347,0.393415,0.237932,0.376864,0.225619,0.253732,1.0,0.253809,0.157115,0.096694,0.089668,0.106564
2,123,0.631347,0.395085,0.236262,0.374220,0.223700,0.252519,1.0,0.253809,0.158018,0.095791,0.088767,0.105759


test WER mean: 0.39407991343998766
test WER std : 0.0008851155917595137
mean WER gain: 0.2372671767524539
seeds with WER CI > 0: 3
